<a href="https://colab.research.google.com/github/Khalimovgeek/Kaggle_tesnorflow_trainings/blob/main/News_Category_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow
!pip install tensorflow-datasets


In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models,utils,datasets
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

In [4]:
!cp /content/drive/MyDrive/dataset/train/train.csv /content/train.csv
!cp /content/drive/MyDrive/dataset/test/test.csv /content/test.csv

In [5]:
# train_path = '/content/drive/MyDrive/dataset/train/train.csv'
# test_path = '/content/drive/MyDrive/dataset/test/test.csv'

# train_path = '/content/train.csv'
# test_path = '/content/test.csv'

In [31]:

# train_dataset = tf.data.experimental.make_csv_dataset(train_path,batch_size=32,label_name='Class Index',num_parallel_reads=tf.data.AUTOTUNE)
# test_dataset = tf.data.experimental.make_csv_dataset(test_path,batch_size=32,label_name='Class Index',num_parallel_reads=tf.data.AUTOTUNE)
import pandas as pd
import tensorflow as tf

train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

train_text = (
    train_df["Title"] + " " + train_df["Description"]
).values

train_labels = train_df["Class Index"].values - 1

test_text = (
    test_df["Title"] + " " + test_df["Description"]
).values

test_labels = test_df["Class Index"].values - 1

train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_text, train_labels)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_text, test_labels)
)

train_dataset = (
    train_dataset
    .shuffle(10000)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

In [32]:
for text, label in train_dataset.take(1):
    print(text[0].numpy().decode())
    print(label[0].numpy())

Nortel Says Canada Plans Accounting Probe Nortel Networks Corp. said on Monday the Royal Canadian Mounted Police told\the company it will begin a criminal investigation into
3


In [33]:
MAX_TOKENS = 10000
SEQUNCE_LEN = 200
vectorize_layer = layers.TextVectorization(max_tokens=MAX_TOKENS,output_mode='int',output_sequence_length=SEQUNCE_LEN)


In [34]:
print(train_dataset.cardinality())

tf.Tensor(3750, shape=(), dtype=int64)


In [35]:
import pandas as pd

df = pd.read_csv(train_path)
print(df.shape)

(120000, 3)


In [37]:
import time

start = time.time()

count = 0
for text, label in train_dataset:
    count += 1

print("Batches:", count)
print("Time:", time.time() - start)

Batches: 3750
Time: 1.8717334270477295


In [36]:
train_text = train_dataset.map(lambda text, label: text)
vectorize_layer.adapt(train_text)

In [41]:
print(f"Vocabulary size after adaptation: {len(vectorize_layer.get_vocabulary())}")
def create_model():

    vocab_size = len(vectorize_layer.get_vocabulary())

    model = models.Sequential([
        keras.Input(shape=(), dtype=tf.string),
        vectorize_layer,

        layers.Embedding(vocab_size, 64, mask_zero=True, input_length=SEQUNCE_LEN),

        layers.Conv1D(128, 5, padding='same', activation='relu'),
        layers.Dropout(0.2),
        layers.MaxPooling1D(),

        layers.Conv1D(256, 5, padding='same', activation='relu'),
        layers.Dropout(0.2),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.5),

        layers.Dense(128, activation='relu'),
        layers.Dense(4, activation='softmax')
    ])

    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return model

Vocabulary size after adaptation: 10000


In [42]:
my_model = create_model()
my_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'conv1d_4' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_4            │ (None, 200)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 200, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 200, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 100, 256)       │       164,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 878,596 (3.35 MB)

 Trainable params: 878,596 (3.35 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = my_model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset
)

Epoch 1/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 521s 138ms/step - accuracy: 0.8789 - loss: 0.3410 - val_accuracy: 0.9064 - val_loss: 0.2920
Epoch 2/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 519s 138ms/step - accuracy: 0.9243 - loss: 0.2265 - val_accuracy: 0.9079 - val_loss: 0.2807
Epoch 3/10
3544/3750 ━━━━━━━━━━━━━━━━━━━━ 28s 137ms/step - accuracy: 0.9269 - loss: 0.2168